# 导出模型1 -> ONNX（kline-de-pre 可直接加载）

CPU 导出，不需要 GPU。输出 `transformer_with_feat_merged.onnx`（`dct_output[9]`+`feat_output[128]`）到 `/kaggle/working`。

In [ ]:
import os
if not os.path.exists('kline-de-pre-model'):
    !git clone --depth 1 https://github.com/lin7c/kline-de-pre-model.git
%cd kline-de-pre-model
!pip -q install onnx onnxscript   # onnxscript: torch 新版 onnx 导出所需
import torch, onnx
print('torch', torch.__version__, '| onnx', onnx.__version__)

In [ ]:
# 导出（CPU）：模型1 + 模型2
!python export_onnx.py checkpoints/transformer_dct_v1.pth /kaggle/working/transformer_with_feat_merged.onnx
!python export_onnx.py --diffusion checkpoints/diffusion_delta_v1.pth /kaggle/working/diffusion_unet_merged.onnx

# 校验 ONNX 的输入输出契约
import onnx, os
for f in ['transformer_with_feat_merged.onnx', 'diffusion_unet_merged.onnx']:
    p = '/kaggle/working/' + f
    if not os.path.exists(p):
        print('❌ 缺少', f); continue
    m = onnx.load(p)
    onnx.checker.check_model(m)
    print(f)
    print('  inputs :', [(i.name, [d.dim_value or d.dim_param for d in i.type.tensor_type.shape.dim]) for i in m.graph.input])
    print('  outputs:', [(o.name, [d.dim_value or d.dim_param for d in o.type.tensor_type.shape.dim]) for o in m.graph.output])
    print('  size KB:', os.path.getsize(p)//1024)